# 🛒 Pandas Assignment — E-Commerce Sales Analysis
### Olist Brazilian E-Commerce Dataset

Welcome to the practical assignment for the Pandas Complete Guide. You will perform a full end-to-end analysis of a real e-commerce company's sales data, applying every pandas technique covered in the lesson notebook.

---

## 🎯 Learning Objectives
By the end of this assignment you will have:
- **Loaded and merged** 7+ related CSV tables (orders, customers, products, sellers, payments, items, reviews)
- **Handled missing values** with multiple strategies based on context
- **Joined and merged** DataFrames using every join type (inner, left, right, outer)
- **Performed group-by aggregations** for business KPIs
- **Conducted time-series analysis** including resampling, rolling windows, and trend detection
- **Produced a final business report** with concrete, data-backed insights

---

## 📦 The Dataset

We use the **Olist Brazilian E-Commerce Public Dataset** — 100,000+ real (anonymized) orders from 2016–2018, organized across 9 interrelated tables. This is one of the most popular publicly available sales datasets and is widely used in data science portfolios.

### Schema overview
```
                  ┌─────────────┐
                  │  customers  │
                  └──────┬──────┘
                         │ customer_id
                  ┌──────▼──────┐
        ┌────────►│   orders    │◄────────┐
        │         └──────┬──────┘         │
        │ order_id       │ order_id       │ order_id
┌───────┴──────┐  ┌──────▼─────┐   ┌──────┴────────┐
│   reviews    │  │ order_items │   │   payments   │
└──────────────┘  └─┬─────────┬─┘   └──────────────┘
                    │         │
              product_id   seller_id
                    │         │
              ┌─────▼───┐ ┌───▼────┐
              │ products│ │ sellers│
              └─────────┘ └────────┘
```

### 📥 How to obtain the dataset

**Option A — Real Olist data (recommended for portfolio work):**
1. Visit https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
2. Sign in to Kaggle (free) and click **Download** to get `archive.zip` (~45 MB)
3. Extract and place all CSV files into a folder named `olist_data/` in the same directory as this notebook
4. Set `USE_REAL_DATA = True` in the setup cell below

**Option B — Synthetic equivalent (run immediately):**
- Leave `USE_REAL_DATA = False` (default)
- The notebook will generate a smaller synthetic dataset with the same schema and column names
- All assignment questions and code work identically on both datasets

> **Tip:** start with Option B to build your code. If you want to extend the work into a portfolio piece, swap to Option A — your code will keep working on the larger real dataset.

---

## 📝 How to Submit

Fill in every code cell marked `# Your code here`. Add markdown cells with your interpretation wherever a question asks **"explain"** or **"comment"**.

Each section has its own **rubric** at the top — check yourself before moving on.

> ⏱ **Estimated time:** 6–10 hours of focused work depending on depth of exploration.

---
## 🔧 Setup

Run these cells first. They import libraries and either load the real Olist files or generate synthetic equivalents.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (11, 5)

# ────────────────────────────────────────────────────────────────────
# SWITCH BETWEEN REAL OLIST DATA AND SYNTHETIC DATA
# ────────────────────────────────────────────────────────────────────
USE_REAL_DATA = True   # set to True if you've downloaded Olist from Kaggle
DATA_DIR = 'dataset/'  # folder containing the Olist CSVs (real data only)

print(f"USE_REAL_DATA = {USE_REAL_DATA}")
print(f"pandas = {pd.__version__}")

USE_REAL_DATA = True
pandas = 2.1.4


### Synthetic data generator (only runs if `USE_REAL_DATA = False`)

This builds 7 CSV files with the same column names as the real Olist dataset. Records are interrelated and span 2 years so the time-series questions are meaningful.

In [3]:
import os

if not USE_REAL_DATA:
    rng = np.random.default_rng(42)
    os.makedirs('synthetic_olist', exist_ok=True)

    # ── 1. CUSTOMERS ────────────────────────────────────────────────
    n_customers = 3000
    states = ['SP','RJ','MG','RS','PR','SC','BA','DF','GO','PE','CE','PA','MA','MT','MS']
    state_p = [0.42,0.13,0.12,0.06,0.05,0.04,0.04,0.03,0.03,0.02,0.02,0.01,0.01,0.01,0.01]
    cities = {
        'SP':['sao paulo','campinas','santos','ribeirao preto','sorocaba'],
        'RJ':['rio de janeiro','niteroi','duque de caxias'],
        'MG':['belo horizonte','uberlandia','contagem','juiz de fora'],
        'RS':['porto alegre','caxias do sul','pelotas'],
        'PR':['curitiba','londrina','maringa'],
        'SC':['florianopolis','joinville','blumenau'],
        'BA':['salvador','feira de santana'],
        'DF':['brasilia'],
        'GO':['goiania','aparecida de goiania'],
        'PE':['recife','olinda'],
        'CE':['fortaleza','caucaia'],
        'PA':['belem','ananindeua'],
        'MA':['sao luis'],
        'MT':['cuiaba','varzea grande'],
        'MS':['campo grande'],
    }
    cust_states = rng.choice(states, n_customers, p=state_p)
    customers = pd.DataFrame({
        'customer_id': [f'cust_{i:06d}' for i in range(n_customers)],
        'customer_unique_id': [f'uniq_{rng.integers(0, n_customers*0.85):06d}' for _ in range(n_customers)],
        'customer_zip_code_prefix': rng.integers(1000, 99999, n_customers),
        'customer_city': [rng.choice(cities[s]) for s in cust_states],
        'customer_state': cust_states,
    })
    customers.to_csv('synthetic_olist/olist_customers_dataset.csv', index=False)

    # ── 2. SELLERS ──────────────────────────────────────────────────
    n_sellers = 250
    sel_states = rng.choice(states, n_sellers, p=state_p)
    sellers = pd.DataFrame({
        'seller_id': [f'sell_{i:04d}' for i in range(n_sellers)],
        'seller_zip_code_prefix': rng.integers(1000, 99999, n_sellers),
        'seller_city': [rng.choice(cities[s]) for s in sel_states],
        'seller_state': sel_states,
    })
    sellers.to_csv('synthetic_olist/olist_sellers_dataset.csv', index=False)

    # ── 3. PRODUCTS ────────────────────────────────────────────────
    n_products = 800
    categories = ['cama_mesa_banho','beleza_saude','esporte_lazer','informatica_acessorios',
                  'moveis_decoracao','utilidades_domesticas','relogios_presentes','telefonia',
                  'automotivo','brinquedos','cool_stuff','ferramentas_jardim','perfumaria',
                  'bebes','eletronicos','livros_interesse_geral']
    products = pd.DataFrame({
        'product_id': [f'prod_{i:05d}' for i in range(n_products)],
        'product_category_name': rng.choice(categories, n_products),
        'product_name_lenght': rng.integers(20, 80, n_products).astype(float),
        'product_description_lenght': rng.integers(100, 4000, n_products).astype(float),
        'product_photos_qty': rng.integers(1, 8, n_products).astype(float),
        'product_weight_g': rng.integers(50, 30000, n_products).astype(float),
        'product_length_cm': rng.integers(10, 100, n_products).astype(float),
        'product_height_cm': rng.integers(2, 80, n_products).astype(float),
        'product_width_cm': rng.integers(5, 80, n_products).astype(float),
    })
    # Inject realistic missingness
    for col in ['product_category_name','product_weight_g','product_length_cm','product_height_cm','product_width_cm']:
        miss_idx = rng.choice(n_products, size=int(n_products*0.03), replace=False)
        products.loc[miss_idx, col] = np.nan
    products.to_csv('synthetic_olist/olist_products_dataset.csv', index=False)

    # ── 4. ORDERS ────────────────────────────────────────────────
    n_orders = 7000
    # Build datetime distribution: increasing volume over time + weekly seasonality
    start = pd.Timestamp('2017-01-01')
    days_range = 730
    # weight days so volume grows over time and dips on weekends slightly
    day_weights = []
    for d in range(days_range):
        date = start + pd.Timedelta(days=d)
        base = 0.5 + d/days_range  # linear growth
        if date.weekday() in [5, 6]:
            base *= 0.85  # slightly lower weekend volume
        # holiday spike around November (Black Friday)
        if date.month == 11 and 20 <= date.day <= 28:
            base *= 2.5
        day_weights.append(base)
    day_weights = np.array(day_weights) / sum(day_weights)
    chosen_days = rng.choice(days_range, n_orders, p=day_weights)
    purchase_ts = [start + pd.Timedelta(days=int(d), hours=int(rng.integers(8,23)),
                                         minutes=int(rng.integers(0,60))) for d in chosen_days]
    purchase_ts = pd.to_datetime(purchase_ts)

    statuses = rng.choice(['delivered','shipped','canceled','invoiced','processing'],
                           n_orders, p=[0.92, 0.03, 0.03, 0.01, 0.01])
    approved_ts = [t + pd.Timedelta(hours=int(rng.integers(0,48))) for t in purchase_ts]
    carrier_ts = [t + pd.Timedelta(days=int(rng.integers(1,7))) for t in approved_ts]
    delivered_ts = [t + pd.Timedelta(days=int(rng.integers(2,15))) for t in carrier_ts]
    estimated_ts = [t + pd.Timedelta(days=int(rng.integers(10,25))) for t in purchase_ts]

    orders = pd.DataFrame({
        'order_id': [f'ord_{i:06d}' for i in range(n_orders)],
        'customer_id': rng.choice(customers['customer_id'], n_orders),
        'order_status': statuses,
        'order_purchase_timestamp': purchase_ts,
        'order_approved_at': approved_ts,
        'order_delivered_carrier_date': carrier_ts,
        'order_delivered_customer_date': delivered_ts,
        'order_estimated_delivery_date': estimated_ts,
    })
    # Inject missingness: non-delivered orders have missing delivery dates
    orders.loc[orders['order_status'] != 'delivered', 'order_delivered_customer_date'] = pd.NaT
    orders.loc[orders['order_status'].isin(['canceled','processing','invoiced']),
               'order_delivered_carrier_date'] = pd.NaT
    # A few approved_at missing too
    miss_idx = rng.choice(n_orders, size=int(n_orders*0.005), replace=False)
    orders.loc[miss_idx, 'order_approved_at'] = pd.NaT
    orders.to_csv('synthetic_olist/olist_orders_dataset.csv', index=False)

    # ── 5. ORDER ITEMS ────────────────────────────────────────────────
    item_rows = []
    for i, oid in enumerate(orders['order_id']):
        n_items = rng.choice([1, 1, 1, 1, 2, 2, 3], p=[0.55,0.15,0.05,0.05,0.12,0.05,0.03])
        purchase_t = orders.iloc[i]['order_purchase_timestamp']
        for item_n in range(1, n_items + 1):
            price = round(float(rng.lognormal(4.3, 0.7)), 2)
            freight = round(price * float(rng.uniform(0.05, 0.25)), 2)
            item_rows.append({
                'order_id': oid,
                'order_item_id': item_n,
                'product_id': rng.choice(products['product_id']),
                'seller_id': rng.choice(sellers['seller_id']),
                'shipping_limit_date': purchase_t + pd.Timedelta(days=int(rng.integers(2, 8))),
                'price': price,
                'freight_value': freight,
            })
    order_items = pd.DataFrame(item_rows)
    order_items.to_csv('synthetic_olist/olist_order_items_dataset.csv', index=False)

    # ── 6. PAYMENTS ──────────────────────────────────────────────────
    payment_rows = []
    for oid in orders['order_id']:
        n_pay = rng.choice([1, 1, 1, 2], p=[0.92, 0.04, 0.02, 0.02])
        order_total = order_items[order_items['order_id'] == oid][['price','freight_value']].sum().sum()
        for seq in range(1, n_pay + 1):
            pay_type = rng.choice(['credit_card','boleto','voucher','debit_card'], p=[0.74,0.19,0.05,0.02])
            installments = rng.choice([1,2,3,4,5,6,8,10], p=[0.5,0.12,0.10,0.08,0.06,0.05,0.05,0.04]) if pay_type=='credit_card' else 1
            payment_rows.append({
                'order_id': oid,
                'payment_sequential': seq,
                'payment_type': pay_type,
                'payment_installments': int(installments),
                'payment_value': round(float(order_total / n_pay), 2),
            })
    payments = pd.DataFrame(payment_rows)
    payments.to_csv('synthetic_olist/olist_order_payments_dataset.csv', index=False)

    # ── 7. REVIEWS ──────────────────────────────────────────────────
    review_rows = []
    for i, oid in enumerate(orders['order_id']):
        if rng.random() < 0.85:  # 85% have a review
            score = rng.choice([1,2,3,4,5], p=[0.10, 0.05, 0.10, 0.20, 0.55])
            create_ts = orders.iloc[i]['order_purchase_timestamp'] + pd.Timedelta(days=int(rng.integers(5,25)))
            answer_ts = create_ts + pd.Timedelta(days=int(rng.integers(0,10))) if rng.random() < 0.7 else pd.NaT
            review_rows.append({
                'review_id': f'rev_{i:06d}',
                'order_id': oid,
                'review_score': int(score),
                'review_comment_title': rng.choice(['Excelente','Bom','Recomendo','Péssimo','Otimo', None],
                                                   p=[0.15,0.15,0.10,0.05,0.10,0.45]),
                'review_comment_message': rng.choice(['Produto chegou rapido','Recomendo a todos',
                                                       'Produto com defeito','Atrasou a entrega',
                                                       'Otima qualidade', None],
                                                      p=[0.18,0.12,0.05,0.05,0.15,0.45]),
                'review_creation_date': create_ts,
                'review_answer_timestamp': answer_ts,
            })
    reviews = pd.DataFrame(review_rows)
    reviews.to_csv('synthetic_olist/olist_order_reviews_dataset.csv', index=False)

    # ── 8. CATEGORY TRANSLATION ──────────────────────────────────────
    translation = pd.DataFrame({
        'product_category_name': categories,
        'product_category_name_english': [
            'bed_bath_table','health_beauty','sports_leisure','computers_accessories',
            'furniture_decor','housewares','watches_gifts','telephony',
            'auto','toys','cool_stuff','garden_tools','perfumery',
            'baby','electronics','books_general_interest'
        ],
    })
    translation.to_csv('synthetic_olist/product_category_name_translation.csv', index=False)

    DATA_DIR = 'synthetic_olist/'
    print('✅ Synthetic dataset generated in:', DATA_DIR)
    print('Files written:')
    for f in sorted(os.listdir(DATA_DIR)):
        print(f'   {f}  ({os.path.getsize(DATA_DIR + f)/1024:.1f} KB)')

### Load all the CSVs

In [4]:
customers      = pd.read_csv(DATA_DIR + 'olist_customers_dataset.csv')
orders         = pd.read_csv(DATA_DIR + 'olist_orders_dataset.csv')
order_items    = pd.read_csv(DATA_DIR + 'olist_order_items_dataset.csv')
products       = pd.read_csv(DATA_DIR + 'olist_products_dataset.csv')
sellers        = pd.read_csv(DATA_DIR + 'olist_sellers_dataset.csv')
payments       = pd.read_csv(DATA_DIR + 'olist_order_payments_dataset.csv')
reviews        = pd.read_csv(DATA_DIR + 'olist_order_reviews_dataset.csv')
translation    = pd.read_csv(DATA_DIR + 'product_category_name_translation.csv')

# Parse datetime columns up front
for col in ['order_purchase_timestamp','order_approved_at',
            'order_delivered_carrier_date','order_delivered_customer_date',
            'order_estimated_delivery_date']:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

reviews['review_creation_date']     = pd.to_datetime(reviews['review_creation_date'])
reviews['review_answer_timestamp']  = pd.to_datetime(reviews['review_answer_timestamp'], errors='coerce')
order_items['shipping_limit_date']  = pd.to_datetime(order_items['shipping_limit_date'])

print('All 8 tables loaded:')
for name, t in [('customers',customers),('orders',orders),('order_items',order_items),
                ('products',products),('sellers',sellers),('payments',payments),
                ('reviews',reviews),('translation',translation)]:
    print(f'  {name:15s} {t.shape}')

All 8 tables loaded:
  customers       (3000, 5)
  orders          (7000, 8)
  order_items     (8593, 7)
  products        (800, 9)
  sellers         (250, 4)
  payments        (7139, 5)
  reviews         (5963, 7)
  translation     (16, 2)


---
# Part 1 — Data Loading & Inspection

> **Rubric:** for each table you should be able to state (a) what one row represents, (b) the primary key, (c) the foreign keys that link to other tables, (d) the data types of each column, (e) the count of missing values per column.

### Q1.1 — Display the first 5 rows of every table.

In [5]:
# Your code here


### Q1.2 — For each table, print `.shape`, `.dtypes`, and `.info()`.

In [6]:
# Your code here


### Q1.3 — Identify keys
For each table, write down (as a markdown comment in a cell below) what the **primary key** is and what **foreign keys** link it to other tables.

For example, in `orders`:
- Primary key: `order_id`
- Foreign keys: `customer_id` → `customers`

In [6]:
# Your code here


### Q1.4 — How many unique customers, sellers, products, and orders are there?
Use `.nunique()` rather than `len()` — they differ if a table has duplicates.

In [7]:
# Your code here


### Q1.5 — What date range does the dataset cover?
Compute the min and max of `order_purchase_timestamp` and the total span in days.

In [8]:
# Your code here


### Q1.6 — Memory usage
Compute the **total memory** used by all 8 DataFrames combined. Which table uses the most memory?

In [9]:
# Your code here


### Q1.7 — Order status distribution
What are the unique values of `order_status` in the orders table, and how many rows fall into each? Sort by count, descending. What proportion of orders were ultimately delivered?

In [10]:
# Your code here


---
# Part 2 — Handling Missing Values

> **Rubric:** for every column that has missing values, you should be able to justify *why* it's missing and *what* you do about it. There is no single right answer — context matters.

### Q2.1 — Build a missing-value summary
Produce a DataFrame with one row per (table, column) pair where missing values exist. Columns: `table`, `column`, `n_missing`, `pct_missing`. Sort descending by `pct_missing`.

**Hint:**
```python
summary = []
for name, t in [('customers', customers), ('orders', orders), ...]:
    miss = t.isna().sum()
    for col, n in miss[miss > 0].items():
        summary.append({'table': name, 'column': col, 'n_missing': n, 'pct_missing': n/len(t)*100})
pd.DataFrame(summary).sort_values('pct_missing', ascending=False)
```

In [11]:
# Your code here


### Q2.2 — Visualize the missing-value pattern in `orders`
Build a heatmap of `orders.isna()`. Comment on what you see.

In [12]:
# Your code here


### Q2.3 — Explain the missingness in `orders.order_delivered_customer_date`
Group the orders DataFrame by `order_status` and count how many rows have `order_delivered_customer_date` missing for each status. Write a one-sentence interpretation: **why** would non-delivered orders have a missing delivery date?

In [13]:
# Your code here


### Q2.4 — Impute `order_approved_at`
A small number of rows have a missing `order_approved_at` despite being delivered. Impute these with the **mean approval delay** (mean of `order_approved_at - order_purchase_timestamp`) added to the purchase timestamp. Verify there are no more missing approved-at values among delivered orders afterwards.

In [14]:
# Your code here


### Q2.5 — Decide what to do about missing `product_category_name`
A few products are missing their category name. Show how many. Pick one of these strategies and apply it:
- (a) Fill with `'unknown'`
- (b) Drop those rows
- (c) Impute based on similar products (advanced)

Justify your choice in a comment.

In [15]:
# Your code here


### Q2.6 — Missing review comments
Reviews have many missing `review_comment_message` values. Is this missingness **informative** (do customers without comments give different scores than customers with comments)? Test this by computing the mean `review_score` for each group.

In [16]:
# Your code here


---
# Part 3 — Merging & Joining Datasets

> **Rubric:** you should choose the right join type for each task, predict the resulting row count before merging, and verify it after.

### Q3.1 — Inner join: orders + customers
Inner-join `orders` with `customers` on `customer_id`. How many rows result? Are there any orphan rows (orders without a customer, or vice versa)? Use `indicator=True` to check.

In [17]:
# Your code here


### Q3.2 — Left join: order_items + products
Left-join `order_items` with `products` on `product_id`. After joining, how many rows have a missing `product_category_name`? Were these missing before the join, or are they introduced by the join itself?

In [18]:
# Your code here


### Q3.3 — Translate categories to English
Left-join the result of Q3.2 with `translation` to add an English category name. Verify that no rows were lost.

In [19]:
# Your code here


### Q3.4 — Build the master sales DataFrame
Build a single wide DataFrame by sequentially merging:

`order_items` → `orders` → `customers` → `products` → `sellers` → `translation`

This is your **master analytical DataFrame** — every subsequent question uses it. Name it `sales`.

**Hint:** chain `.merge()` calls. Use `how='left'` since you want to keep every item.

In [20]:
# Your code here


### Q3.5 — Add total payment per order
Aggregate `payments` to get one row per `order_id` (sum `payment_value`, count `payment_sequential`), then merge into `sales`.

### Q3.6 — Add review score per order
Many orders have multiple reviews (rarely, but it happens). Aggregate `reviews` to one row per order (take the **mean** review_score) and merge into `sales`. Comment on how many rows of `sales` end up with a missing `review_score` after this — what does that tell you?

In [22]:
# Your code here


### Q3.7 — `concat` practice
Use `pd.concat` to vertically stack the `customers` and `sellers` tables into one "people" DataFrame with a new column `role` indicating which they were. Make sure the column names align before concatenating.

In [23]:
# Your code here


---
# Part 4 — GroupBy Operations & Business KPIs

> **Rubric:** you should compute meaningful business metrics, present them cleanly (sorted, formatted), and add a brief interpretation for each.

### Q4.1 — Total revenue by product category
Compute total revenue (sum of `price`) by `product_category_name_english`. Display the top 10 categories. Use a horizontal bar chart.

In [24]:
# Your code here


### Q4.2 — Revenue and order count by customer state
For each `customer_state`, compute total revenue, number of orders, and average order value. Sort by total revenue descending. Top 10.

In [25]:
# Your code here


### Q4.3 — Two-level group-by: state × category
Create a pivot table where rows are `customer_state`, columns are top 5 `product_category_name_english`, and values are total revenue. Use `fillna(0)` to clean up.

In [26]:
# Your code here


### Q4.4 — Top 10 sellers by revenue
Using `groupby('seller_id')`, find the 10 sellers with highest total revenue. Include their state and category mix (count of distinct categories sold).

In [27]:
# Your code here


### Q4.5 — Top 10 customers by spending
Same idea for customers. Be careful: in Olist, every order has a different `customer_id`. Use `customer_unique_id` to track repeat customers.

In [28]:
# Your code here


### Q4.6 — Multi-statistic aggregation with named agg
For each product category, compute in one `.agg()` call:
- `n_orders`: count of distinct orders
- `n_items`: count of items sold
- `total_revenue`: sum of price
- `avg_price`: mean of price
- `median_price`: median of price
- `total_freight`: sum of freight_value
- `avg_review_score`: mean review score

Sort by `total_revenue` desc. Top 10.

In [29]:
# Your code here


### Q4.7 — Payment behavior
For each `payment_type`, find:
- The number of payments
- The mean `payment_value`
- The mean `payment_installments`

Add an interpretation: which payment type is associated with the highest-value orders?

In [30]:
# Your code here


### Q4.8 — `transform` — deviation from category mean
For every row in `sales`, add a column `price_vs_category_avg` showing how that item's price compares to the average price in its category. Then list the 10 most "overpriced" items (largest positive deviation).

In [31]:
# Your code here


---
# Part 5 — Time Series Analysis

> **Rubric:** you should be able to resample to different frequencies, smooth with rolling windows, decompose trend vs. seasonality, and present time series visually.

### Q5.1 — Daily order volume
Resample `orders` to daily frequency (count of orders per day). Plot the result as a line chart. Comment on overall trend and any obvious anomalies.

In [32]:
# Your code here


### Q5.2 — Monthly revenue and orders
Build a monthly summary DataFrame with columns: `month`, `n_orders`, `total_revenue`, `avg_order_value`. Plot revenue over time.

**Hint:**
```python
sales['month'] = sales['order_purchase_timestamp'].dt.to_period('M').dt.to_timestamp()
monthly = sales.groupby('month').agg(n_orders=('order_id','nunique'),
                                      total_revenue=('price','sum'),
                                      avg_order_value=('price','mean')).reset_index()
```

In [33]:
# Your code here


### Q5.3 — Rolling 7-day average of daily orders
Compute and plot the 7-day rolling mean of daily order count on top of the raw daily counts. Smoothing reveals the underlying trend.

In [34]:
# Your code here


### Q5.4 — Day-of-week seasonality
On which day of the week do most orders happen? Use `dt.day_name()`, group, count, and plot.

In [35]:
# Your code here


### Q5.5 — Hour-of-day seasonality
At what hour of the day do customers shop most? Bar chart of order counts by hour (0–23).

In [36]:
# Your code here


### Q5.6 — Month-of-year seasonality
Aggregate across all years — for each calendar month (Jan–Dec), what is the total order count? Are there spikes (e.g. Black Friday in November)?

In [37]:
# Your code here


### Q5.7 — Year-over-year growth
Compare monthly revenue in 2017 to monthly revenue in 2018 for matching months. Compute YoY growth rate for each month where both years have data.

In [38]:
# Your code here


### Q5.8 — Delivery time analysis
For delivered orders, compute the actual delivery time in days as `order_delivered_customer_date - order_purchase_timestamp`. Then compute:
- Mean and median delivery time
- Distribution histogram
- Mean delivery time **by customer state** — top 5 slowest states

Then compute the **on-time delivery rate**: the % of orders delivered before their `order_estimated_delivery_date`.

In [39]:
# Your code here


### Q5.9 — Time between purchase and review
For each review, compute `review_creation_date - order_purchase_timestamp` in days. Plot the distribution. Most customers leave a review how long after purchase?

In [40]:
# Your code here


### Q5.10 — Cumulative revenue plot
Plot **cumulative revenue** over time (running total). This is a classic CFO chart — the steeper the curve, the more the business is growing.

In [41]:
# Your code here


---
# Part 6 — Advanced Analyses

These tie everything together. They are open-ended — emphasize *insight* and clean presentation.

### Q6.1 — Repeat customer rate
What proportion of `customer_unique_id` values appear in more than one order? Among repeat customers, what is the average number of orders?

In [42]:
# Your code here


### Q6.2 — RFM analysis
Compute Recency, Frequency, and Monetary values for each `customer_unique_id`:
- **Recency:** days since their most recent purchase (relative to the dataset's max date)
- **Frequency:** number of orders
- **Monetary:** total spending

Then assign each customer a 1–5 score on each dimension using `pd.qcut`. Combine into an RFM segment label like `'5-5-5'` (best customers). How many customers are in the top segment `'5-5-5'`?

In [43]:
# Your code here


### Q6.3 — Does delivery delay hurt reviews?
Compute the relationship between delivery delay (`actual delivery date - estimated delivery date`, in days, where positive = late) and `review_score`. Group orders into "early/on-time" and "late" and compare mean review scores. Is the difference statistically significant? (Use `scipy.stats.ttest_ind`.)

In [44]:
# Your code here


### Q6.4 — Category × payment method
Build a cross-tab showing the proportion of orders in each `product_category_name_english` that used `credit_card` vs. `boleto` vs. other methods. Which categories see the most installment usage?

In [45]:
# Your code here


### Q6.5 — Cohort retention (advanced — optional)
Define a "cohort" as the month a customer made their first purchase. For each cohort, what proportion of those customers made another purchase 1, 2, 3, ..., 12 months later? Visualize as a heatmap.

In [46]:
# Your code here


---
# Part 7 — Final Business Report

Write a markdown cell below summarizing your findings. Imagine you're presenting to the executive team of an e-commerce company.

### Required sections:

1. **Executive Summary** (3–4 sentences) — overall state of the business
2. **Top 3 Insights** — with one supporting chart or statistic each
3. **Top 3 Recommendations** — concrete, data-backed actions
4. **Caveats & Limitations** — what would you want more data on?

### Grading criteria
- Insights are backed by specific numbers, not vague statements
- Each insight references a specific section/question
- Recommendations are actionable (not "improve customer service" but "extend delivery network in state X to reduce mean delivery time from N to M days")
- Writing is concise — bullet points are fine

## ✍️ Your Final Report — write below

### Executive Summary

*(write here)*

### Top 3 Insights

**1.** *(insight 1 with supporting data)*

**2.** *(insight 2 with supporting data)*

**3.** *(insight 3 with supporting data)*

### Top 3 Recommendations

**1.** *(action 1)*

**2.** *(action 2)*

**3.** *(action 3)*

### Caveats & Limitations

*(what would you investigate with more data?)*

---
## 🎓 Reflection (optional but encouraged)

Briefly answer below:
1. Which pandas concept did this assignment force you to use that you found least intuitive?
2. Which part felt most like "real" data work?
3. If you used the synthetic data option — what would you do differently with the real Olist dataset?

> **Submission:** once complete, run `Kernel → Restart and Run All` to verify every cell executes top-to-bottom without errors, then export to PDF or share the notebook directly.

---

### 📚 Further Practice
- Replace the analysis with the full Olist dataset (Kaggle) and see how your numbers change at scale.
- Add a **forecast**: use the monthly revenue series to predict the next 3 months with a simple model (e.g. linear regression on time, or `statsmodels.SARIMAX`).
- Build a **dashboard** in Streamlit or Plotly Dash from your final report.